# **BART и BCF: байесовские деревья для эффекта**

Практика к модулю [«Гетерогенные эффекты»](https://ai-interpretability.school/learn/5/0/0), урок «Байесовские деревья: BART и BCF».

В уроке «Байесовские деревья: BART и BCF» BART ([Chipman, George и McCulloch, 2010](https://doi.org/10.1214/09-AOAS285)) был суммой многих маленьких деревьев с априорным распределением, а [Hill (2011)](https://doi.org/10.1198/jcgs.2010.08162) предложила оценивать им эффект: обучить одну модель исхода на признаках и воздействии и взять разность прогнозов, как в S-learner. Байесовский причинный лес, BCF ([Hahn, Murray и Carvalho, 2020](https://arxiv.org/abs/1706.09523)), разделяет модель на две суммы деревьев: прогностическую $\mu(x)$ и эффект $\tau(x)$ со своим, более строгим априорным распределением, и добавляет оценку склонности $\hat\pi(x)$ как признак. Здесь мы проверим три утверждения урока:

1. апостериорное распределение дает каждому объекту интервал для эффекта без дополнительных процедур;
2. при целевом отборе — когда воздействие чаще получают объекты с худшим прогнозом без воздействия — регуляризация смещает оценку эффекта, а признак $\hat\pi(x)$ это смещение уменьшает;
3. раздельные модели для двух групп, как в T-learner, дают худшую оценку, чем одна модель с явной функцией эффекта.

План тетради:

1. данные Hahn et al.: нелинейный прогноз, неоднородный эффект, целевой отбор;
2. BART с воздействием как признаком (Hill, 2011);
3. оценка склонности и BART с ней как с признаком;
4. BCF и сравнение четырех схем;
5. где ломается: смещение из-за регуляризации и интервалы одной цепи MCMC;
6. реализации.

Тетрадь идет на процессоре; весь прогон занимает четыре-шесть минут, почти все время — повторы в разделе 5.1.

In [ ]:
!pip install -q stochtree==0.4.5

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from stochtree import BARTModel, BCFModel

SEED = 0
MCMC = dict(num_gfr=10, num_burnin=200, num_mcmc=500)   # 10 жадных проходов для старта, в каждой цепи 200 отброшенных и 500 сохраненных шагов MCMC
CHAINS = 4                                               # четыре независимые цепи MCMC, всего 2000 выборок
pd.set_option('display.width', 140)

## 1. Данные Hahn et al.

Повторяем схему раздела 6.1 статьи BCF: 500 объектов, пять признаков — три непрерывных $x_1,x_2,x_3\sim\mathcal N(0,1)$, бинарный $x_4$ и категориальный $x_5\in\{1,2,3\}$. Нелинейный прогноз без воздействия и неоднородный эффект:

$$\mu(x)=-6+g(x_5)+6\,|x_3-1|,\qquad \tau(x)=1+2\,x_2x_5,$$

где $g(1)=2$, $g(2)=-1$, $g(3)=-4$. В тексте статьи $g$ стоит при бинарном $x_4$, но значений у $g$ три, как уровней $x_5$; мы применяем $g$ к $x_5$, а $x_4$ остается признаком, от которого ничего не зависит. Склонность — целевой отбор: вероятность воздействия растет вместе с $\mu(x)$,

$$\pi(x)=0.8\,\Phi\bigl(3\mu(x)/s-0.5\,x_1\bigr)+0.05+u/10,\qquad u\sim U(0,1),$$

где $s$ — стандартное отклонение $\mu$ по выборке. Исход $Y=\mu(x)+\tau(x)\,Z+\varepsilon$, $\varepsilon\sim\mathcal N(0,1)$. Категориальный $x_5$ кодируем тремя индикаторами, а в $\tau$ он входит своим номером.

In [ ]:
def simulate(n, r, heterogeneous=True):
    x = r.normal(size=(n, 3))
    x4 = r.binomial(1, 0.5, n)
    x5 = r.integers(1, 4, n)
    mu = -6 + np.select([x5 == 1, x5 == 2], [2.0, -1.0], -4.0) + 6 * np.abs(x[:, 2] - 1)
    tau = 1 + 2 * x[:, 1] * x5 if heterogeneous else np.full(n, 3.0)
    pi = 0.8 * norm.cdf(3 * mu / mu.std() - 0.5 * x[:, 0]) + 0.05 + r.uniform(size=n) / 10
    Z = r.binomial(1, pi)
    Y = mu + tau * Z + r.normal(size=n)
    X = np.column_stack([x, x4, x5 == 1, x5 == 2, x5 == 3]).astype(float)
    return X, Z, Y, tau, pi, mu


X, Z, Y, tau, pi, mu = simulate(500, np.random.default_rng(SEED))
print('объектов:', len(Y), '| с воздействием:', Z.sum())
print('склонность: от', round(pi.min(), 3), 'до', round(pi.max(), 3), '| корреляция склонности с mu(x):', round(np.corrcoef(pi, mu)[0, 1], 3))
print('истинный средний эффект:', round(tau.mean(), 3), '| стандартное отклонение эффекта:', round(tau.std(), 3))
print('наивная разность средних исходов:', round(Y[Z == 1].mean() - Y[Z == 0].mean(), 3))

Наивная разность средних далеко от истинного среднего эффекта: воздействие получают объекты с большим $\mu(x)$, и разность групп смешивает эффект с различием прогнозов.

## 2. BART с воздействием как признаком (Hill, 2011)

Одна сумма деревьев $f(x,z)$ учится на признаках и воздействии вместе. Каждый сохраненный шаг MCMC дает одну выборку функции $f$ из апостериорного распределения, и для объекта $i$ эффект в этой выборке — $f(x_i,1)-f(x_i,0)$. Запускаем четыре независимые цепи по 500 сохраненных шагов; по 2000 выборкам получаем апостериорное распределение эффекта каждого объекта: среднее — точечная оценка, 2.5- и 97.5-процентные квантили — 95-процентный интервал. Средний эффект по выборке — то же для среднего по объектам.

В `stochtree` модель `BARTModel().sample(X, y, ...)` возвращает выборки прогноза через `predict(X, terms='y_hat')` — матрицу «объекты $\times$ выборки». `random_seed` в `general_params` фиксирует генератор MCMC, `num_chains` задает число цепей.

**Задание 1. Напишите функцию `bart_s(Xa, Z, Y)`: она обучает `BARTModel` на признаках `Xa` со столбцом воздействия, а возвращает матрицу выборок эффекта: прогноз при $Z=1$ минус прогноз при $Z=0$. Напишите функцию `summarize(draws, tau)`, которая возвращает ошибку оценки среднего эффекта (апостериорное среднее минус истинное), покрыт ли истинный средний эффект 95-процентным интервалом, RMSE оценки CATE по объектам (апостериорные средние против истинных $\tau(x_i)$), долю объектов, у которых интервал накрыл $\tau(x_i)$, и среднюю ширину интервала. Чему равен RMSE оценки CATE у BART?**

In [ ]:
# Ваш код здесь

**Ваш ответ:**

## 3. Оценка склонности как признак

Hahn et al. показывают, почему BART с воздействием как признаком смещается при целевом отборе. Априорное распределение тянет функцию к простой форме. Если склонность почти монотонна по $\mu(x)$, то столбец $Z$ сам по себе хорошо предсказывает исход: простая модель может приписать воздействию часть различия в $\mu(x)$. Авторы называют это смещением из-за регуляризации (regularization-induced confounding, RIC). Лекарство — дать модели признак, который прямо несет эту информацию: оценку склонности $\hat\pi(x)$.

Склонность оценим тоже суммой деревьев — регрессией $Z$ на $X$, как это делает `stochtree` внутри `BCFModel`, когда склонность не передана. Оценку ограничим отрезком $[0.01, 0.99]$.

In [ ]:
def propensity(X, Z, seed=SEED):
    m = BARTModel()
    m.sample(X, Z.astype(float), num_gfr=10, num_burnin=0, num_mcmc=10, general_params={'random_seed': seed, 'num_threads': 1})
    return np.clip(m.predict(X, terms='y_hat', type='mean'), 0.01, 0.99)


pihat = propensity(X, Z)
print('корреляция оценки склонности с истинной:', round(np.corrcoef(pihat, pi)[0, 1], 3))
draws['ps-BART'] = bart_s(np.column_stack([X, pihat]), Z, Y)
print(pd.Series(summarize(draws['ps-BART'], tau)).round(3))

## 4. BCF и сравнение четырех схем

BCF записывает исход как $Y=\mu(x,\hat\pi(x))+\tau(x)\,Z+\varepsilon$: две суммы деревьев с разными априорными распределениями. У прогностической $\mu$ — 250 деревьев (так по умолчанию в `stochtree`; в статье Hahn et al. — 200) с обычным для BART априорным распределением ($\eta=0.95$, $\beta=2$ в вероятности расщепления $\eta\,(1+d)^{-\beta}$ узла глубины $d$). У эффекта $\tau$ — меньше деревьев и $\eta=0.25$, $\beta=3$: деревья почти не растут, и априорное распределение тянет эффект к одинаковому для всех. В статье у $\tau$ 50 деревьев, у `stochtree` по умолчанию 100; берем 50, как в статье.

Для сравнения добавим четвертую схему — две отдельные BART для групп воздействия и контроля, аналог T-learner (в статье — BART $(f_0,f_1)$).

**Задание 2 (для самопроверки: на сайте вопроса к нему нет — MCMC на другом процессоре идет по другой траектории, и число может отличаться от нашего: при сидах 0–4 оно лежит от 1.28 до 1.31). Обучите `BCFModel` с `propensity_train=pihat` и `treatment_effect_forest_params={'num_trees': 50}`; выборки эффекта — `predict(X, Z, propensity=pihat, terms='tau')`. Обучите две отдельные `BARTModel` на объектах с $Z=1$ и с $Z=0$ и возьмите разность их прогнозов для всех объектов. Сведите `summarize` для четырех схем в таблицу. Чему равен RMSE оценки CATE у двух раздельных BART?**

In [ ]:
# Ваш код здесь

**Ваш ответ:**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, name in zip(axes, ['BART', 'BCF']):
    c = draws[name].mean(axis=1)
    lo, hi = np.quantile(draws[name], [0.025, 0.975], axis=1)
    order = np.argsort(tau)
    ax.fill_between(range(len(tau)), lo[order], hi[order], color='#9ecae1', alpha=0.6, label='95% posterior interval')
    ax.plot(range(len(tau)), c[order], '.', ms=3, color='#3b6ea5', label='Posterior mean')
    ax.plot(range(len(tau)), tau[order], color='#c0392b', lw=1.5, label='True effect')
    ax.set_title(name)
    ax.set_xlabel('Objects sorted by true effect')
    ax.set_ylabel('Effect')
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 5. Где ломается

### 5.1. Смещение из-за регуляризации

Одна выборка — один розыгрыш шума. Чтобы увидеть смещение, а не случайность, повторим расчет на 10 выборках с одинаковым для всех эффектом $\tau(x)=3$, где RIC виден в чистом виде: любое отклонение оценки среднего эффекта от 3 — это ошибка, а не неоднородность. На каждой выборке обучим три схемы: BART без $\hat\pi$, BCF без $\hat\pi$ (`propensity_covariate='none'`) и BCF с $\hat\pi$.

**Задание 3. Для выборок `simulate(500, np.random.default_rng(100 + k), heterogeneous=False)`, $k=0,\dots,9$, посчитайте для трех схем среднюю по выборкам ошибку оценки среднего эффекта, ее RMSE и долю выборок, где 95-процентный интервал среднего эффекта накрыл 3. Как признак $\hat\pi$ меняет ошибку?** Ответ сверяйте по числу выборок из 10, где интервал накрыл 3: у BART и BCF без $\hat\pi$ при сидах MCMC 0–4 оно не меняется, а у BCF с $\hat\pi$ средняя ошибка при разных сидах лежит от 0.021 до 0.036.

In [ ]:
# Ваш код здесь

**Ваш ответ:**

### 5.2. Одна цепь MCMC

Интервалы BART — квантили выборок из апостериорного распределения, и они верны настолько, насколько цепь MCMC это распределение обошла. Сумма деревьев перестраивается медленно: одна цепь может долго ходить вокруг одного набора деревьев, и тогда разброс выборок меньше настоящей апостериорной неопределенности. Независимые цепи стартуют из разных жадных проходов и это видят.

**Задание 4. Обучите BART из раздела 2 одной цепью (`chains=1`, тот же сид) и сравните с четырьмя цепями: RMSE оценки CATE, долю объектов, у которых 95-процентный интервал накрыл $\tau(x_i)$, и среднюю ширину интервала. Затем для четырех цепей посчитайте ту же долю объектов с накрытым $\tau(x_i)$ отдельно по выборкам каждой цепи (выборки цепи $c$ — столбцы с $500c$ по $500(c+1)-1$). Что происходит с интервалами одной цепи?**

In [ ]:
# Ваш код здесь

**Ваш ответ:**

## 6. Реализации

- [`stochtree`](https://github.com/StochasticTree/stochtree) (лицензия MIT; Python и R): `BARTModel` и `BCFModel`, как в тетради. Сэмплер по умолчанию начинает с нескольких жадных проходов (grow-from-root, параметр `num_gfr`) и продолжает обычным MCMC Chipman et al.; вариант BCF в пакете — с параметризацией кодирования воздействия из раздела 5.3 статьи Hahn et al.
- [`pymc-bart`](https://github.com/pymc-devs/pymc-bart) (лицензия Apache-2.0): BART как распределение внутри модели PyMC; эффект собирается вручную, как в разделе 2.
- R-пакет `bcf` авторов BCF и пакеты BART для R в тетради не используются.

## Что унести из тетради

- **BART дает интервал каждому объекту даром,** но это апостериорный интервал модели, а не гарантия покрытия, и одна цепь MCMC делает его уже, чем нужно.
- **При целевом отборе регуляризация смещает оценку эффекта,** и признак $\hat\pi(x)$ уменьшает это смещение — у BART и у BCF.
- **Отдельные модели для двух групп работают хуже одной общей:** у каждой меньше данных, и простота эффекта не используется.
- **BCF задает эффекту свое априорное распределение,** поэтому сжатие к одинаковому эффекту можно настроить, не трогая модель прогноза.

Источники:
1. [Chipman, George и McCulloch, 2010. BART: Bayesian additive regression trees](https://doi.org/10.1214/09-AOAS285).
2. [Hill, 2011. Bayesian nonparametric modeling for causal inference](https://doi.org/10.1198/jcgs.2010.08162).
3. [Hahn, Murray и Carvalho, 2020. Bayesian regression tree models for causal inference: regularization, confounding, and heterogeneous effects](https://arxiv.org/abs/1706.09523).
4. [stochtree](https://github.com/StochasticTree/stochtree).